In [1]:
import networkx as nx

In [2]:
g = nx.DiGraph()

In [3]:
def fetch_context(path:str):
    return ((d["keyword"], g.nodes[u]["value"]) for u, v, d in g.in_edges(path, data=True))

def consumers(path:str):
    return (v for u,v in g.out_edges(path))

def feeders(path:str):
    return (u for u,v in g.in_edges(path))

def register_change(path:str, value:str, order:dict[str, int]|None=None):
    node = g.nodes[path]
    order = order or _chain(path)
    if node["value"] != value:
        node["value"] = value
        children = sorted(consumers(path), key=lambda x:order[x])
        for childPath in children:
            print(childPath)
            f = g.nodes[childPath]
            temp = f["func"]
            target = f["target"]
            kw = dict(fetch_context(childPath))
            v = temp.format(**kw)
            register_change(target, v, order)
            
    
def _chain(path:str, order:int=0, collector:dict[str, int]|None=None):
    """
    DFS walk down all paths from a parent node keeping track of the longest
    path length from the source.
    """
    collector = collector or {}
    for (u, v) in g.out_edges(path):
        collector[v] = max(collector.get(v, 0), order + 1)
        _chain(v, order+1, collector)
    return collector

def insert_node(path:str, value:str):
    g.add_node(path, value=value, type="data")

def insert_func(path:str, func:str, context:dict[str, str], target:str):
    assert len(list(g.in_edges(target))) == 0
    g.add_node(path, func=func, target=target)
    for k,v in context.items():
        g.add_edge(v, path, keyword=k)
        g.add_edge(path, target)
    assert nx.is_directed_acyclic_graph(g)


In [4]:
g.clear()
insert_node("/a", "")
insert_node("/b", "")
insert_node("/c", "")
insert_node("/d", "")
insert_func(
    "/functions/C", 
    '{a}, {c}', 
    {"a": "/a", "c": "/c"}, 
    "/d"
)
insert_func("/functions/A", '{a}', {"a": "/a"}, "/b")
insert_func("/functions/B", '{b}', {"b": "/b"}, "/c")

In [5]:
register_change("/a", "=")

/functions/A
/functions/B
/functions/C
/functions/C


In [6]:
for n, d in g.nodes(data=True):
    if "value" in d:
        print(n, d["value"])

/a =
/b =
/c =
/d =, =
